<a href="https://colab.research.google.com/github/Vidogreg/nlp-summer-school-2026/blob/main/demos/whisper/speech_to_text_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Whisper Speech-to-Text demo (Colab)

Transcribes speech with [openai/whisper-large-v3-turbo](https://huggingface.co/openai/whisper-large-v3-turbo) (English and Slovak), then translates it into several languages with [facebook/nllb-200-distilled-600M](https://huggingface.co/facebook/nllb-200-distilled-600M). Caches both models to Google Drive, same as the OmniVoice notebook, so you don't re-download them every session.

In [ ]:
# Colab already ships a working torch/CUDA pair, so only install what's missing.
!pip install -q -U transformers accelerate soundfile pydub torchaudio datasets

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

DRIVE_CACHE_DIR = "/content/drive/MyDrive/whisper_cache"
os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
os.environ["HF_HOME"] = DRIVE_CACHE_DIR

In [ ]:
import torch
from transformers import pipeline

ASR_MODEL_ID = "openai/whisper-large-v3-turbo"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device.startswith("cuda") else torch.float32
pipeline_device = 0 if device.startswith("cuda") else -1

try:
    asr = pipeline(
        "automatic-speech-recognition",
        model=ASR_MODEL_ID,
        torch_dtype=dtype,
        device=pipeline_device,
    )
except Exception as e:
    print(f"Drive cache unavailable ({e!r}); falling back to a fresh download.")
    os.environ.pop("HF_HOME", None)
    asr = pipeline(
        "automatic-speech-recognition",
        model=ASR_MODEL_ID,
        torch_dtype=dtype,
        device=pipeline_device,
    )

## Sanity check

Transcribes one public sample clip with a known transcript, to confirm the model works before moving to the mic.

In [ ]:
from datasets import load_dataset

sample = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")[0]
result = asr(sample["audio"])

print("Reference:  ", sample["text"])
print("Transcribed:", result["text"])

In [ ]:
from google.colab.output import eval_js
from IPython.display import Javascript, Audio, display
from base64 import b64decode
from pydub import AudioSegment
import io
import numpy as np
import torchaudio

_RECORD_JS = """
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = e => resolve(e.srcElement.result);
  reader.readAsDataURL(blob);
});
var record = time => new Promise(async (resolve, reject) => {
  try {
    const micTimeout = new Promise((_, rej) => setTimeout(
      () => rej(new Error("Timed out waiting for microphone access — check for a mic permission prompt or blocked-mic icon in the address bar.")),
      15000
    ));
    const stream = await Promise.race([
      navigator.mediaDevices.getUserMedia({ audio: true }),
      micTimeout,
    ]);
    const recorder = new MediaRecorder(stream);
    const chunks = [];
    recorder.ondataavailable = e => chunks.push(e.data);
    recorder.start();
    await new Promise(r => setTimeout(r, time));
    recorder.onstop = async () => {
      stream.getTracks().forEach(t => t.stop());
      resolve(await b2text(new Blob(chunks)));
    };
    recorder.stop();
  } catch (err) {
    reject(err.message || String(err));
  }
});
"""

def record_audio(seconds=6):
    """Record from the browser mic, returns (waveform float32 numpy, sample_rate)."""
    display(Javascript(_RECORD_JS))
    data_url = eval_js(f"record({seconds * 1000})")
    raw = b64decode(data_url.split(",", 1)[1])

    segment = AudioSegment.from_file(io.BytesIO(raw)).set_channels(1)
    waveform = np.array(segment.get_array_of_samples()).astype(np.float32)
    waveform /= 1 << (8 * segment.sample_width - 1)
    return waveform, segment.frame_rate

def to_16k(waveform, sr):
    """Whisper expects 16kHz audio."""
    if sr == 16000:
        return waveform
    resampled = torchaudio.functional.resample(
        torch.from_numpy(waveform), orig_freq=sr, new_freq=16000
    )
    return resampled.numpy()

## Transcribe: English

Run the next cell, then say a sentence in English (~6 seconds).

In [ ]:
waveform, sr = record_audio(6)
display(Audio(waveform, rate=sr))

result = asr(
    {"array": to_16k(waveform, sr), "sampling_rate": 16000},
    generate_kwargs={"language": "english", "task": "transcribe"},
)
print("Transcribed:", result["text"])

## Transcribe: Slovak

Run the next cell, then say a sentence in Slovak (~6 seconds).

In [ ]:
waveform, sr = record_audio(6)
display(Audio(waveform, rate=sr))

result = asr(
    {"array": to_16k(waveform, sr), "sampling_rate": 16000},
    generate_kwargs={"language": "slovak", "task": "transcribe"},
)
print("Transcribed:", result["text"])

## Translate into several languages

Records your speech (English or Slovak), transcribes it in the source language with Whisper, then translates that text into several languages with NLLB-200.

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MT_MODEL_ID = "facebook/nllb-200-distilled-600M"
mt_tokenizer = AutoTokenizer.from_pretrained(MT_MODEL_ID)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL_ID, torch_dtype=dtype).to(device)

# NLLB-200 (FLORES-200) language codes: https://github.com/facebookresearch/flores/blob/main/flores200/README.md
SOURCE_LANGUAGES = {"english": "eng_Latn", "slovak": "slk_Latn"}
TARGET_LANGUAGES = {
    "German": "deu_Latn",
    "French": "fra_Latn",
    "Spanish": "spa_Latn",
    "Czech": "ces_Latn",
    "Ukrainian": "ukr_Cyrl",
}

def translate(text, src_lang, tgt_lang):
    mt_tokenizer.src_lang = src_lang
    inputs = mt_tokenizer(text, return_tensors="pt").to(device)
    tokens = mt_model.generate(
        **inputs,
        forced_bos_token_id=mt_tokenizer.convert_tokens_to_ids(tgt_lang),
        max_new_tokens=200,
    )
    return mt_tokenizer.batch_decode(tokens, skip_special_tokens=True)[0]

In [ ]:
SOURCE_LANGUAGE = "english"  # or "slovak"

waveform, sr = record_audio(6)
display(Audio(waveform, rate=sr))

result = asr(
    {"array": to_16k(waveform, sr), "sampling_rate": 16000},
    generate_kwargs={"language": SOURCE_LANGUAGE, "task": "transcribe"},
)
source_text = result["text"]
print(f"Transcribed ({SOURCE_LANGUAGE}): {source_text}\n")

src_code = SOURCE_LANGUAGES[SOURCE_LANGUAGE]
for name, code in TARGET_LANGUAGES.items():
    print(f"{name}: {translate(source_text, src_code, code)}")

## Common ASR failure modes

Even strong models like Whisper turbo struggle with:

- **Named entities** — personal and place names, especially ones from a different language than the speech (e.g. Slovak surnames spoken in an English sentence, or vice versa).
- **Company/brand/product names** — anything newer, niche, or that sounds like an ordinary word, since the model has no domain knowledge to disambiguate from acoustics alone.
- **Numbers, dates, currency, units** — spoken-to-written conversion is inherently ambiguous ("twenty twenty-six" vs "2026"). This is exactly what the DER/DSER metrics in the `slovak-llm-audio` benchmark measure separately from word-level CER/WER.
- **Code-switching** — switching languages mid-sentence (e.g. a Slovak speaker dropping in English technical terms), since most models assume one language per utterance.
- **Low-resource languages** — bigger accuracy gaps for languages with less training data, like Slovak vs English — the reason a dedicated Slovak benchmark exists at all.
- **Homophones & rare words** — usually disambiguated by context, but that safety net is weaker for uncommon vocabulary.
- **Background noise, overlapping speech, multiple speakers** — crosstalk and noisy environments degrade accuracy sharply, and go beyond ASR into diarization.
- **Accents and dialects** — non-native or regional accents underrepresented in training data.
- **Hallucination on silence** — a well-documented Whisper-specific failure: near-silent audio can produce fabricated text instead of an empty transcript.
- **Disfluencies** — filler words, stutters, false starts; models may transcribe them literally, drop them inconsistently, or get confused by them.
- **Punctuation & capitalization** — there's no single "correct" punctuation for spoken language, so this is a common source of apparent errors that aren't really wrong words.